# 01. Фильтрация сырых данных

Правила очистки **сырых** файлов прайса до любого джойна.

| # | Фильтр | Тип | Объект |
|---|--------|-----|--------|
| F1 | `month_span == 'begin'` | удаление строк | прайс |
| F2 | Числовые аномалии: `price > 1`, `area ≥ 1` | удаление строк | прайс |
| F2a | `project_class`: исключение `Премиум` и `Де-люкс` | удаление строк | прайс |
| F2b | `premises_type`: нормализация + исключение `Пентхаус` | нормализация / удаление | прайс |
| F2c | `room_count`: `0→1`, `ст→1`; `'-'` и нетиповые → исключение; нечисловые → исключение; `> 6` → исключение | нормализация / удаление | прайс |
| F2d | `unit_typology`: значения не из `UNIT_TYPOLOGY_ALLOWED` → удаление строк | удаление строк | прайс |
| F2e | `stage_k`, `contract_type_k`, `finish_tier`: `"-"` → NaN; `finish_in_report`: `'Н/Д'` → NaN | нормализация | прайс |
| F2f | `section`: `"-"` → NaN | нормализация | прайс |
| F2g | `floor`: нечисловые + `floor < 1` → удаление строк | удаление строк | прайс |
| F2h | `has_budget != 1` → удаление строк | удаление строк | прайс |
| F2i | `finish_text`: нормализация в 5 групп (`нет_отделки` / `отделка` / `whitebox` / `дизайнерская` / `прочее`) | нормализация | прайс |


In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / 'data').is_dir() else _cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import RAW_DIR


def _report(label: str, before: int, after: int) -> None:
    lost = before - after
    pct  = 100 * lost / before if before else 0
    print(f'  {label:<55s}  {before:>10,} → {after:>10,}  (−{lost:>8,} / {pct:5.1f}%)')

## 0. Константы фильтрации

Все пороги в одном месте. Менять только здесь — все диагностики пересчитаются.

In [2]:
# F1: прайс
MONTH_SPAN_VALUE   = 'begin'

# F2: числовые аномалии
PRICE_MIN          = 1                # руб. за кв.м.
AREA_MIN           = 1.0              # кв.м.

# F2a: класс проекта
# Исключаем нетипичные сегменты: мало наблюдений, иная ценовая динамика
PROJECT_CLASS_EXCLUDE = {'премиум', 'де-люкс'}
PROJECT_CLASS_COL_CANDIDATES = ('Класс проекта', 'project_class')

# F2b: тип помещения
PREMISES_TYPE_ALIASES = {
    'Апартаменты': 'Апартамент',
}
PREMISES_TYPE_EXCLUDE = {'Пентхаус'}
EXCLUDE_APARTMENTS = False

# F2c: комнатность
ROOM_COUNT_REMAP = {
    '0':  '1',
    'ст': '1',
}
ROOM_COUNT_NONTYPICAL = {
    '-',
    'пх', 'сх', 'тх', 'вилла', 'резиденция', 'ситихаус', 'особняк', 'хл',
    'таун', 'уник', 'ОКН', 'дп', 'special', 'Вилла',
}
ROOM_COUNT_MAX = 6

# F2d: тип кв/ап
# '-' нормализуется в NaN до проверки; NaN-строки не удаляются
UNIT_TYPOLOGY_ALLOWED = {'Евро', 'Студия', 'Квартира', 'Пентхаус', 'Два уровня', 'Классика'}

# F2e: нормализация прочерков и служебных значений
DASH_VALUES = {'-', '—', '–'}
DASH_COLS   = ['stage_k', 'contract_type_k', 'finish_tier',
               'Стадия К', 'Договор К', 'Отделка К']
FINISH_IN_REPORT_ND = {'Н/Д'}

# F2f: секция
# '-' (27.4%) = нет секции → NaN (используем DASH_VALUES)

# F2g: этаж
FLOOR_MIN = 1

# F2h: бюджет
# Оставляем только строки с заполненным бюджетом (has_budget == 1 / True)
HAS_BUDGET_COL_CANDIDATES = ('Наличие бюджета', 'has_budget')

print('Константы фильтрации загружены.')

Константы фильтрации загружены.


## 1. Загрузка сырых данных

In [3]:
PRICE_CSV_SLICES = {
    '2020_2025':  'price_dataflat_2020_2025.csv',
    '2023_01_06': 'price_dataflat_2023_01_06.csv',
    '2023_04_12': 'price_dataflat_2023_04_12.csv',
    '2023_07_12': 'price_dataflat_2023_07_12.csv',
    '2024_01_06': 'price_dataflat_2024_01_06.csv',
    '2024_07_12': 'price_dataflat_2024_07_12.csv',
    '2025_01_05': 'price_dataflat_2025_01_05.csv',
}

price_parts = []
for label, fname in PRICE_CSV_SLICES.items():
    path = RAW_DIR / fname
    if not path.is_file():
        print(f'  skip: {fname}')
        continue
    part = pd.read_csv(path, low_memory=False)
    price_parts.append(part)

price_raw = pd.concat(price_parts, ignore_index=True)
N_PRICE_RAW = len(price_raw)

print(f'Сырых строк прайса:  {N_PRICE_RAW:>10,}')

Сырых строк прайса:   5,106,109


## F1 — `month_span == 'begin'`
Оставляем только срез начала месяца, чтобы не задваивать лоты.

In [4]:
ms_col = next((c for c in ('Начало/конец месяца', 'month_span') if c in price_raw.columns), None)

if ms_col:
    print(f'Колонка month_span: {ms_col!r}')
    print('Распределение значений:')
    print(price_raw[ms_col].value_counts(dropna=False).to_string())
    price_begin = price_raw[price_raw[ms_col] == MONTH_SPAN_VALUE].copy()
else:
    print('ВНИМАНИЕ: колонка month_span не найдена — фильтр не применён')
    price_begin = price_raw.copy()

print()
_report('F1  month_span == begin', N_PRICE_RAW, len(price_begin))

Колонка month_span: 'Начало/конец месяца'
Распределение значений:
Начало/конец месяца
begin    5040087
end        66022

  F1  month_span == begin                                   5,106,109 →  5,040,087  (−  66,022 /   1.3%)


## F2 — числовые аномалии (`price`, `area`)
Удаляем строки с `price = 0` / `area = 0` — технический мусор, искажающий цену за м² и лаги.

In [5]:
price_col = next((c for c in ('Цена', 'price') if c in price_begin.columns), None)
area_col  = next((c for c in ('Площадь', 'area') if c in price_begin.columns), None)

n_before = len(price_begin)
mask_ok = pd.Series(True, index=price_begin.index)

anomaly_counts = {}

if price_col:
    price_num = pd.to_numeric(price_begin[price_col], errors='coerce')
    bad_price = price_num.isna() | (price_num <= PRICE_MIN)
    anomaly_counts['price ≤ 0 или null'] = int(bad_price.sum())
    mask_ok &= ~bad_price

if area_col:
    area_num = pd.to_numeric(price_begin[area_col], errors='coerce')
    bad_area = area_num.isna() | (area_num < AREA_MIN)
    anomaly_counts['area < 1 или null'] = int(bad_area.sum())
    mask_ok &= ~bad_area

price_clean = price_begin[mask_ok].copy()

print('Аномалии (могут пересекаться):')
for reason, cnt in anomaly_counts.items():
    print(f'  {reason:<40s}  {cnt:>8,}  ({100*cnt/n_before:.2f}%)')
print()
_report('F2  числовые аномалии (price, area)', n_before, len(price_clean))

Аномалии (могут пересекаться):
  price ≤ 0 или null                          18,979  (0.38%)
  area < 1 или null                              103  (0.00%)

  F2  числовые аномалии (price, area)                       5,040,087 →  5,021,108  (−  18,979 /   0.4%)


## F2a — класс проекта
Убираем «Премиум» и «Де-люкс»: малочисленны и с иной ценовой динамикой.

In [6]:
pc_col = next((c for c in PROJECT_CLASS_COL_CANDIDATES if c in price_clean.columns), None)

N_BEFORE_F2A = len(price_clean)

if pc_col:
    print('Распределение project_class до фильтрации:')
    print(price_clean[pc_col].value_counts(dropna=False).to_string())

    mask_excl = price_clean[pc_col].astype(str).str.casefold().isin(PROJECT_CLASS_EXCLUDE)
    n_excl    = int(mask_excl.sum())
    price_clean = price_clean[~mask_excl].copy()

    print(f'\nИсключено {PROJECT_CLASS_EXCLUDE}: {n_excl:,} строк')
    print('\nПосле фильтрации:')
    print(price_clean[pc_col].value_counts(dropna=False).to_string())
else:
    print(f'ВНИМАНИЕ: колонка project_class не найдена. Ожидалась одна из: {PROJECT_CLASS_COL_CANDIDATES}')

print()
_report('F2a  project_class: исключение премиум/де-люкс', N_BEFORE_F2A, len(price_clean))

Распределение project_class до фильтрации:
Класс проекта
комфорт    3904173
бизнес      994070
премиум      84991
де-люкс      37874

Исключено {'премиум', 'де-люкс'}: 122,865 строк

После фильтрации:
Класс проекта
комфорт    3904173
бизнес      994070

  F2a  project_class: исключение премиум/де-люкс            5,021,108 →  4,898,243  (− 122,865 /   2.4%)


## F2b — тип помещения
Нормализуем опечатки (`Апартаменты→Апартамент`), убираем `Пентхаус`; апартаменты оставляем как признак (`EXCLUDE_APARTMENTS=False`).

In [7]:
pt_col = next((c for c in ('Тип помещения', 'premises_type') if c in price_clean.columns), None)

n_before = len(price_clean)

if pt_col:
    print('До нормализации:')
    print(price_clean[pt_col].value_counts(dropna=False).to_string())

    # Нормализация опечаток
    price_clean[pt_col] = price_clean[pt_col].replace(PREMISES_TYPE_ALIASES)

    # Исключение 'Пентхаус' из premises_type (это тип кв/ап, не тип помещения)
    mask_excl = price_clean[pt_col].isin(PREMISES_TYPE_EXCLUDE)
    n_excl = mask_excl.sum()
    price_clean = price_clean[~mask_excl].copy()

    print(f'\nИсключено PREMISES_TYPE_EXCLUDE {PREMISES_TYPE_EXCLUDE}: {n_excl:,} строк')

    if EXCLUDE_APARTMENTS:
        mask_apt = price_clean[pt_col].isin({'Апартамент'})
        price_clean = price_clean[~mask_apt].copy()
        print(f'EXCLUDE_APARTMENTS=True: исключено {mask_apt.sum():,} строк апартаментов')
    else:
        print('EXCLUDE_APARTMENTS=False: апартаменты сохранены как признак')

    print('\nПосле нормализации:')
    print(price_clean[pt_col].value_counts(dropna=False).to_string())
else:
    print('ВНИМАНИЕ: колонка premises_type не найдена')

print()
_report('F2b  premises_type нормализация/фильтр', n_before, len(price_clean))

До нормализации:
Тип помещения
Квартира       4477817
Апартамент      420423
Апартаменты          3

Исключено PREMISES_TYPE_EXCLUDE {'Пентхаус'}: 0 строк
EXCLUDE_APARTMENTS=False: апартаменты сохранены как признак

После нормализации:
Тип помещения
Квартира      4477817
Апартамент     420426

  F2b  premises_type нормализация/фильтр                    4,898,243 →  4,898,243  (−       0 /   0.0%)


## F2c — комнатность
`0`/`ст` → `1`; убираем нетиповые (`пх`, `вилла`…), нечисловые и `room_count > 6`.

In [8]:
rc_col = next((c for c in ('Комнатность', 'room_count') if c in price_clean.columns), None)

n_before = len(price_clean)

if rc_col:
    print('До нормализации (топ-15):')
    print(price_clean[rc_col].astype(str).value_counts(dropna=False).head(15).to_string())

    # Нормализация: перекодировка значений
    price_clean[rc_col] = price_clean[rc_col].astype(str).replace(ROOM_COUNT_REMAP)

    # Удаление нетиповых текстовых значений
    mask_nontypical = price_clean[rc_col].isin(ROOM_COUNT_NONTYPICAL)
    n_nontypical    = mask_nontypical.sum()

    # Удаление нечисловых значений, оставшихся после нормализации
    rc_num = pd.to_numeric(price_clean[rc_col], errors='coerce')
    mask_not_numeric = rc_num.isna() & ~mask_nontypical
    n_not_numeric    = mask_not_numeric.sum()

    # Удаление числовых выбросов (room_count > MAX)
    mask_over_max = rc_num.notna() & (rc_num > ROOM_COUNT_MAX)
    n_over_max    = mask_over_max.sum()

    mask_drop = mask_nontypical | mask_not_numeric | mask_over_max
    price_clean = price_clean[~mask_drop].copy()

    print(f'\nНормализовано (перекодировка): {len(ROOM_COUNT_REMAP)} правил')
    print(f'Исключено нетиповых значений:  {n_nontypical:>8,}')
    print(f'Исключено нечисловых:          {n_not_numeric:>8,}')
    print(f'Исключено room_count > {ROOM_COUNT_MAX}:     {n_over_max:>8,}')

    print('\nПосле фильтрации (топ-15):')
    print(price_clean[rc_col].astype(str).value_counts(dropna=False).head(15).to_string())
else:
    print('ВНИМАНИЕ: колонка room_count не найдена')

print()
_report('F2c  room_count нормализация + фильтр', n_before, len(price_clean))

До нормализации (топ-15):
Комнатность
2           1692093
1           1468531
0            818152
3            762232
4            149534
2 ур           4161
пх             2768
5               377
ст              133
тх              104
ситихаус         32
сх               28
хл               24
2ур              22
3 ур             21

Нормализовано (перекодировка): 2 правил
Исключено нетиповых значений:     2,956
Исключено нечисловых:             4,208
Исключено room_count > 6:           15

После фильтрации (топ-15):
Комнатность
1    2286816
2    1692093
3     762232
4     149534
5        377
6         12

  F2c  room_count нормализация + фильтр                     4,898,243 →  4,891,064  (−   7,179 /   0.1%)


## F2d — тип кв/ап
`'-'` → NaN (строки остаются); строки с не-NaN значением не из `UNIT_TYPOLOGY_ALLOWED` удаляем.

In [9]:
ut_col = next((c for c in ('Тип кв/ап', 'unit_typology') if c in price_clean.columns), None)

N_BEFORE_F2D = len(price_clean)

if ut_col:
    # Нормализация: '-' и другие прочерки → NaN (строки не удаляются)
    n_dash = int(price_clean[ut_col].astype(str).isin(DASH_VALUES).sum())
    price_clean[ut_col] = price_clean[ut_col].where(
        ~price_clean[ut_col].astype(str).isin(DASH_VALUES), other=np.nan
    )
    print(f'Нормализовано "-" → NaN: {n_dash:,} строк')

    print(f'Допустимые значения (UNIT_TYPOLOGY_ALLOWED): {sorted(UNIT_TYPOLOGY_ALLOWED)}')
    print(f'\nРаспределение после нормализации (до фильтрации):')
    print(price_clean[ut_col].value_counts(dropna=False).to_string())

    # Фильтрация: NaN оставляем, не-NaN не из ALLOWED — удаляем
    mask_allowed = price_clean[ut_col].isna() | price_clean[ut_col].astype(str).isin(UNIT_TYPOLOGY_ALLOWED)
    n_invalid = int((~mask_allowed).sum())

    print(f'\nНедопустимые значения (топ-10):')
    print(price_clean.loc[~mask_allowed, ut_col].value_counts(dropna=False).head(10).to_string())

    price_clean = price_clean[mask_allowed].copy()

    print(f'\nПосле фильтрации:')
    print(price_clean[ut_col].value_counts(dropna=False).to_string())
else:
    print('ВНИМАНИЕ: колонка unit_typology не найдена')

print()
_report('F2d  unit_typology: нормализация "-"→NaN + удаление недопустимых', N_BEFORE_F2D, len(price_clean))

Нормализовано "-" → NaN: 3,111,780 строк
Допустимые значения (UNIT_TYPOLOGY_ALLOWED): ['Два уровня', 'Евро', 'Квартира', 'Классика', 'Пентхаус', 'Студия']

Распределение после нормализации (до фильтрации):
Тип кв/ап
NaN                      3111780
Евро                     1012221
Студия                    754159
Квартира                    6030
Классика                     659
12                           525
13                           482
17                           469
11                           468
16                           461
10                           425
2                            378
9                            376
15                           373
14                           323
6                            323
8                            317
4                            272
3                            259
5                            220
Апартамент                   218
7                            183
0                             56
High Level               

## F2e — прочерки в `stage_k` / `contract_type_k` / `finish_tier`
Прочерки (`-`, `—`, `–`) и `Н/Д` → NaN (нет данных, не категория); строки не удаляем.

In [10]:
dash_found = []

# Прочерки в stage_k / contract_type_k / finish_tier
for col in DASH_COLS:
    actual = next((c for c in price_clean.columns if c == col), None)
    if actual is None:
        continue
    n_dash = price_clean[actual].astype(str).isin(DASH_VALUES).sum()
    dash_found.append((actual, n_dash))
    price_clean[actual] = price_clean[actual].where(
        ~price_clean[actual].astype(str).isin(DASH_VALUES), other=np.nan
    )

if dash_found:
    print('Замена прочерков на NaN:')
    for col, n in dash_found:
        print(f'  {col:<30s}  {n:>8,} прочерков → NaN  ({100*n/len(price_clean):.1f}%)')
else:
    print(f'ВНИМАНИЕ: ни одна из DASH_COLS не найдена. Ожидались: {DASH_COLS}')

# finish_in_report: 'Н/Д' → NaN
fir_col = next((c for c in ('Отделка в отчет', 'finish_in_report') if c in price_clean.columns), None)
if fir_col:
    n_nd = price_clean[fir_col].isin(FINISH_IN_REPORT_ND).sum()
    price_clean[fir_col] = price_clean[fir_col].where(
        ~price_clean[fir_col].isin(FINISH_IN_REPORT_ND), other=np.nan
    )
    print(f'\n  {fir_col:<30s}  {n_nd:>8,} "Н/Д" → NaN  ({100*n_nd/len(price_clean):.1f}%)')
else:
    print('\nВНИМАНИЕ: колонка finish_in_report не найдена')

print()
print('F2e завершён — строки не удалялись')

Замена прочерков на NaN:
  Стадия К                         523,324 прочерков → NaN  (10.7%)
  Договор К                        523,324 прочерков → NaN  (10.7%)
  Отделка К                        523,324 прочерков → NaN  (10.7%)

  Отделка в отчет                  252,787 "Н/Д" → NaN  (5.2%)

F2e завершён — строки не удалялись


## F2f — секция
`'-'` (≈27% строк) → NaN: секция отсутствует/неизвестна, не категория.

In [11]:
sec_col = next((c for c in ('Секция', 'section') if c in price_clean.columns), None)

if sec_col:
    n_dash = int(price_clean[sec_col].astype(str).isin(DASH_VALUES).sum())
    print(f'Прочерков в section: {n_dash:,}  ({100*n_dash/len(price_clean):.1f}%)')

    price_clean[sec_col] = price_clean[sec_col].where(
        ~price_clean[sec_col].astype(str).isin(DASH_VALUES), other=np.nan
    )

    n_na = int(price_clean[sec_col].isna().sum())
    print(f'NaN в section после замены: {n_na:,}  ({100*n_na/len(price_clean):.1f}%)')
else:
    print('ВНИМАНИЕ: колонка section не найдена')

print()
print('F2f завершён — строки не удалялись')

Прочерков в section: 1,324,054  (27.1%)
NaN в section после замены: 1,324,054  (27.1%)

F2f завершён — строки не удалялись


## F2g — этаж
Удаляем строки с нечисловым этажом и `floor < 1` (подземные/нежилые).

In [12]:
fl_col = next((c for c in ('Этаж', 'floor') if c in price_clean.columns), None)

N_BEFORE_F2G = len(price_clean)

if fl_col:
    fl_raw = price_clean[fl_col]
    fl_num = pd.to_numeric(fl_raw, errors='coerce')

    mask_nonnumeric = fl_raw.notna() & fl_num.isna()
    mask_low        = fl_num.notna() & (fl_num < FLOOR_MIN)
    mask_drop       = mask_nonnumeric | mask_low

    n_nonnumeric = int(mask_nonnumeric.sum())
    n_low        = int(mask_low.sum())

    if n_nonnumeric > 0:
        print(f'Нечисловые значения floor (топ-20):')
        print(price_clean.loc[mask_nonnumeric, fl_col].value_counts().head(20).to_string())
    else:
        print('Нечисловых значений floor не обнаружено.')

    if n_low > 0:
        print(f'\nЗначения floor < {FLOOR_MIN}:')
        print(price_clean.loc[mask_low, fl_col].value_counts().head(10).to_string())
    else:
        print(f'\nЗначений floor < {FLOOR_MIN} не обнаружено.')

    price_clean = price_clean[~mask_drop].copy()

    # Привести к целому — чтобы в unit_match_key было "5", а не "5.0"
    price_clean[fl_col] = pd.to_numeric(price_clean[fl_col], errors='coerce').round().astype('Int64')

    print(f'\nУдалено нечисловых: {n_nonnumeric:>8,}  ({100*n_nonnumeric/N_BEFORE_F2G:.2f}%)')
    print(f'Удалено floor < {FLOOR_MIN}:  {n_low:>8,}  ({100*n_low/N_BEFORE_F2G:.2f}%)')
    print(f'Dtype floor после каста: {price_clean[fl_col].dtype}')
else:
    print('ВНИМАНИЕ: колонка floor не найдена')

print()
_report('F2g  floor: нечисловые + floor < 1', N_BEFORE_F2G, len(price_clean))

Нечисловые значения floor (топ-20):
Этаж
-         3220
19-19А     172
2-3        112
P1          32
1A          31
P2          26
             6
19А          4

Значения floor < 1:
Этаж
-1    440
0     410
-2     77

Удалено нечисловых:    3,603  (0.07%)
Удалено floor < 1:       927  (0.02%)
Dtype floor после каста: Int64

  F2g  floor: нечисловые + floor < 1                        4,884,849 →  4,880,319  (−   4,530 /   0.1%)


## F2i — текст отделки
Свободный текст → 5 групп (`нет_отделки` / `whitebox` / `отделка` / `дизайнерская` / `прочее`); `'-'`/пусто → NaN; строки не удаляем.

In [13]:
ft_col = next((c for c in ('Отделка текст', 'finish_text') if c in price_clean.columns), None)

if ft_col:
    print('До нормализации (топ-20):')
    print(price_clean[ft_col].value_counts(dropna=False).head(20).to_string())

    def _classify_finish_text(val):
        if pd.isna(val):
            return np.nan
        s = str(val).strip()
        if s in DASH_VALUES or s == '':
            return np.nan
        sl = s.lower()
        if any(k in sl for k in ('whitebox', 'white box', 'вайтбокс', 'предчистов', 'wb')):
            return 'whitebox'
        if any(k in sl for k in ('без отд', 'б/о', 'черновая', 'без от')):
            return 'нет_отделки'
        if any(k in sl for k in ('дизайнер', 'авторск', 'элит')):
            return 'дизайнерская'
        if any(k in sl for k in ('с отделкой', 'с/о', 'чистовая', 'чистов', 'финиш', 'стандарт', 'отделка', 'с отд')):
            return 'отделка'
        return 'прочее'

    price_clean[ft_col] = price_clean[ft_col].apply(_classify_finish_text)

    print('\nПосле нормализации:')
    print(price_clean[ft_col].value_counts(dropna=False).to_string())
else:
    print('ВНИМАНИЕ: колонка finish_text не найдена')

print()
print('F2i завершён — строки не удалялись')

До нормализации (топ-20):
Отделка текст
-                          2482464
Без отделки                 757105
С отделкой                  353194
готовая                     212777
готовая отделка             106204
нет                          97620
white box                    73650
White-box, Дизайнерская      72274
1                            66812
Дизайнерская                 65507
Чистовая отделка             56552
Чистовая                     47618
mr base                      47480
предчистовая                 45798
whitebox                     45299
Отделка White box            39020
Предчистовая отделка         28514
мебель                       23886
есть                         23532
Возможна отделка             18779

После нормализации:
Отделка текст
NaN             2482464
нет_отделки      773933
прочее           627557
отделка          605794
whitebox         251827
дизайнерская     138744

F2i завершён — строки не удалялись


## F2h — бюджет
Оставляем только строки с `has_budget == 1` (без бюджета нет ценовой информации).

In [14]:
hb_col = next((c for c in HAS_BUDGET_COL_CANDIDATES if c in price_clean.columns), None)

N_BEFORE_F2H_BUDGET = len(price_clean)

if hb_col:
    print(f'Колонка: {hb_col!r}')
    print('Распределение значений:')
    print(price_clean[hb_col].value_counts(dropna=False).to_string())

    col_str = price_clean[hb_col].astype(str).str.strip()
    hb_num  = pd.to_numeric(price_clean[hb_col], errors='coerce')
    # Считаем строку «с бюджетом», если числовое значение == 1 ИЛИ текст «Да»
    has_budget = (hb_num == 1) | col_str.isin({'Да', 'да', 'Yes', 'yes', 'True', 'true', '1'})
    mask_no_budget = ~has_budget
    n_no_budget = int(mask_no_budget.sum())
    price_clean = price_clean[~mask_no_budget].copy()

    print(f'\nУдалено строк без бюджета: {n_no_budget:>8,}  '
          f'({100*n_no_budget/N_BEFORE_F2H_BUDGET:.1f}%)')
else:
    print(f'ВНИМАНИЕ: колонка has_budget не найдена. Ожидалась одна из: {HAS_BUDGET_COL_CANDIDATES}')

print()
_report('F2h  has_budget != 1 → удаление', N_BEFORE_F2H_BUDGET, len(price_clean))

Колонка: 'Наличие бюджета'
Распределение значений:
Наличие бюджета
Да    4880319

Удалено строк без бюджета:        0  (0.0%)

  F2h  has_budget != 1 → удаление                           4,880,319 →  4,880,319  (−       0 /   0.0%)


## Сохранение результата

In [16]:
from src.config import INTERIM_DIR, FILTERED_PARQUET

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# parquet требует единый тип на колонку. После конкатенации срезов часть object-колонок
# имеет смешанный python-тип (int+str, напр. 'ID Проекта' = 551 и '551') -> ArrowInvalid.
# Приводим object-колонки к нативному string (NaN -> <NA>); типы пересчитываются ниже в 02.

_obj_cols = price_clean.select_dtypes('object').columns
if len(_obj_cols):
    price_clean[_obj_cols] = price_clean[_obj_cols].astype('string')

price_clean.to_parquet(FILTERED_PARQUET, index=False)
print(f'Сохранено: {FILTERED_PARQUET}')
print(f'Строк: {len(price_clean):,}  |  Колонок: {price_clean.shape[1]}')

/var/folders/4d/y7_mlhmn0kq7bgf3mkkj_hxr0000gn/T/ipykernel_2579/2438372709.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  _obj_cols = price_clean.select_dtypes('object').columns


Сохранено: /Users/adesayan/demand_elasticity/data/interim/price_filtered.parquet
Строк: 4,880,319  |  Колонок: 40


## Итоговый реестр потерь

Сводная таблица: сколько строк теряется на каждом шаге.

In [17]:
print('=' * 75)
print(f'  {"Шаг":<55s}  {"До":>10s}   {"После":>10s}   {"Потеря":>7s}')
print('=' * 75)

def _summary_row(label, before, after):
    if after is None:
        print(f'  {label:<55s}  {before:>10,}')
    else:
        lost = before - after
        pct  = 100 * lost / before if before else 0
        sign = '−' if lost >= 0 else '+'
        print(f'  {label:<55s}  {before:>10,} → {after:>10,}   {sign}{abs(lost):>5,} ({pct:.1f}%)')

_summary_row('Прайс сырой',                                          N_PRICE_RAW,           None)
_summary_row('F1   month_span == begin',                             N_PRICE_RAW,           len(price_begin))
_summary_row('F2   числовые аномалии (price, area)',                 len(price_begin),      len(price_clean))
_summary_row('F2a  project_class: Премиум/Де-люкс → удаление',      N_BEFORE_F2A,          len(price_clean))
_summary_row('F2b  premises_type нормализация/фильтр',               len(price_clean),      len(price_clean))
_summary_row('F2c  room_count нормализация + фильтр',                len(price_clean),      len(price_clean))
_summary_row('F2d  unit_typology: "-"→NaN + удаление недопустимых',  N_BEFORE_F2D,          len(price_clean))
_summary_row('F2e  stage_k/contract_type_k/finish_tier: -→NaN',      len(price_clean),      len(price_clean))
_summary_row('F2f  section: "-" → NaN',                              len(price_clean),      len(price_clean))
_summary_row('F2g  floor: нечисловые + floor < 1',                   N_BEFORE_F2G,          len(price_clean))
_summary_row('F2h  has_budget != 1 → удаление',                      N_BEFORE_F2H_BUDGET,   len(price_clean))
_summary_row('F2i  finish_text: нормализация в 5 групп',             len(price_clean),      len(price_clean))

print('=' * 75)
print()
print('Строки удаляются: F1, F2 (price/area=0), F2a (Премиум/Де-люкс),')
print('                  F2b (PREMISES_TYPE_EXCLUDE + опц. апарт.),')
print('                  F2c (нетиповые room_count), F2d (unit_typology не в ALLOWED),')
print('                  F2g (floor), F2h (нет бюджета).')
print('Нормализация без удаления: F2d ("-"→NaN), F2e, F2f, F2i.')

  Шаг                                                              До        После    Потеря
  Прайс сырой                                               5,106,109
  F1   month_span == begin                                  5,106,109 →  5,040,087   −66,022 (1.3%)
  F2   числовые аномалии (price, area)                      5,040,087 →  4,880,319   −159,768 (3.2%)
  F2a  project_class: Премиум/Де-люкс → удаление            5,021,108 →  4,880,319   −140,789 (2.8%)
  F2b  premises_type нормализация/фильтр                    4,880,319 →  4,880,319   −    0 (0.0%)
  F2c  room_count нормализация + фильтр                     4,880,319 →  4,880,319   −    0 (0.0%)
  F2d  unit_typology: "-"→NaN + удаление недопустимых       4,891,064 →  4,880,319   −10,745 (0.2%)
  F2e  stage_k/contract_type_k/finish_tier: -→NaN           4,880,319 →  4,880,319   −    0 (0.0%)
  F2f  section: "-" → NaN                                   4,880,319 →  4,880,319   −    0 (0.0%)
  F2g  floor: нечисловые + floor < 1   